# Data Processing

In [3]:
1+1

2

In [4]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import os
import pandas as pd
import random
import torch
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, Sampler
import wandb
from tqdm import tqdm
import sys
from pathlib import Path


In [5]:
print(torch.cuda.is_available())       
print(torch.cuda.get_device_name(0)) 

True
NVIDIA GeForce RTX 5090


In [6]:
sys.path.insert(0,r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal')
from evaluate import evaluate_model, make_val_dl


In [7]:
TRAIN = r'C:\Users\jan\.cache\kagglehub\competitions\nml-2026\train\train'
TEST  = r'C:\Users\jan\.cache\kagglehub\competitions\nml-2026\test\test'

### ORIGINAL

In [6]:
from models.nba_baseline import NBATrainer, NBAModel


In [7]:
baseline_trainer = NBATrainer(
    batch_size=32, epochs=10,
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', seed=0,
)


In [8]:
%%capture
baseline_trainer.pipeline()


In [8]:
model_b = NBAModel(4, 2, 32, 8, 12).to('cuda')
model_b.load_state_dict(torch.load(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_baseline.pth',weights_only=True))

val_dl = make_val_dl(baseline_trainer)
evaluate_model(model_b, baseline_trainer, val_dl, name='Baseline GRU',notes='10 epochs, state_dim=32'); 

Baseline GRU: 54.0075


In [9]:
baseline_trainer.get_kaggle_submission(test_dir=TEST)

### GATT TRAINER

In [10]:
from models.nba_gat import NBAGATTrainer, NBAGATModel

In [11]:
gat_trainer = NBAGATTrainer(
    batch_size=64, epochs=50,
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', seed=0,
    state_dim=64, num_heads=4,
)



In [18]:
%%capture
gat_trainer.pipeline()

In [12]:
model_g = NBAGATModel(4, 2, 64, 8, 12, 4).to('cuda')
model_g.load_state_dict(torch.load(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_gat.pth',weights_only=True))

val_dl = make_val_dl(gat_trainer)
evaluate_model(model_g, gat_trainer, val_dl, name='GAT', notes='50 epochs,state_dim=64, 4 heads');

GAT: 23.2694


In [13]:
gat_trainer.get_kaggle_submission(test_dir=TEST)

# Attention

In [14]:
from models.nba_v2 import NBAv2Trainer, NBAv2Model

v2_trainer = NBAv2Trainer(
    batch_size=64, epochs=100,
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', seed=0,
    state_dim=128, num_heads=4, num_layers=2,
)


In [24]:
%%capture
v2_trainer.pipeline()

In [15]:


model_v2 = NBAv2Model(4, 2, 128, 8, 12, 4, 2).to('cuda')
model_v2.load_state_dict(torch.load(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v2.pth', weights_only=True))

val_dl = make_val_dl(v2_trainer)
evaluate_model(model_v2, v2_trainer, val_dl, name='v2 dist-GAT', notes='100 pochs, state_dim=128, 2 layers');

v2 dist-GAT: 9.2611


In [16]:
v2_trainer.get_kaggle_submission(test_dir=TEST)

# Score of 6.  Velocity + GRU + dist-attention  

In [ ]:
from models.nba_v3 import Trainer, NBAv3Model

SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

# Step 1: Trainer just to get mu/sigma — does NOT train
v3_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=30, batch_size=64,
)

# Step 2: Load the saved model weights
model = NBAv3Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v3.pth',weights_only=True
))
model.eval()

    # Step 3: Generate submission (same logic as v2's get_kaggle_submission)
all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    # Normalize positions using training stats
    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v3_trainer.mu) / v3_trainer.sigma

    # Take last 8 frames as context
    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] — normalized predictions

    # Denormalize
    pred = pred * v3_trainer.sigma.to('cuda') + v3_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v3.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v3.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v3.csv


# Score of 4 (+ Ball-relative features + typed edges   ) 

In [ ]:
from models.nba_v6 import Trainer, NBAv6Model

SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

# Step 1: Trainer just to get mu/sigma — does NOT train
v6_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=30, batch_size=64,
)

# Step 2: Load the saved model weights
model = NBAv6Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v6.pth',weights_only=True
))
model.eval()

    # Step 3: Generate submission (same logic as v2's get_kaggle_submission)
all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    # Normalize positions using training stats
    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v6_trainer.mu) / v6_trainer.sigma

    # Take last 8 frames as context
    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] — normalized predictions

    # Denormalize
    pred = pred * v6_trainer.sigma.to('cuda') + v6_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v6.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v6.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v6.csv


# V 12

In [15]:
from models.nba_v12 import Trainer, NBAv12Model

In [11]:

SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

# Step 1: Trainer just to get mu/sigma — does NOT train
v12_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=30, batch_size=64,
)

# Step 2: Load the saved model weights
model = NBAv12Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v12.pth',weights_only=True
))
model.eval()

    # Step 3: Generate submission (same logic as v2's get_kaggle_submission)
all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    # Normalize positions using training stats
    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v12_trainer.mu) / v12_trainer.sigma

    # Take last 8 frames as context
    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] — normalized predictions

    # Denormalize
    pred = pred * v12_trainer.sigma.to('cuda') + v12_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v12.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v12.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v12.csv


# V 15

In [ ]:
from models.nba_v15 import Trainer, NBAv15Model

In [8]:

SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

# Step 1: Trainer just to get mu/sigma — does NOT train
v15_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=30, batch_size=64,
)

# Step 2: Load the saved model weights
model = NBAv15Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v15.pth',weights_only=True
))
model.eval()

    # Step 3: Generate submission (same logic as v2's get_kaggle_submission)
all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    # Normalize positions using training stats
    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v15_trainer.mu) / v15_trainer.sigma

    # Take last 8 frames as context
    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] — normalized predictions

    # Denormalize
    pred = pred * v15_trainer.sigma.to('cuda') + v15_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v15.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v15.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v15.csv


# V 19 


In [ ]:
from models.nba_v19 import Trainer, NBAv19Model

In [14]:

SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

# Step 1: Trainer just to get mu/sigma — does NOT train
v19_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=30, batch_size=64,
)

# Step 2: Load the saved model weights
model = NBAv19Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v19.pth',weights_only=True
))
model.eval()

    # Step 3: Generate submission (same logic as v2's get_kaggle_submission)
all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    # Normalize positions using training stats
    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v19_trainer.mu) / v19_trainer.sigma

    # Take last 8 frames as context
    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] — normalized predictions

    # Denormalize
    pred = pred * v19_trainer.sigma.to('cuda') + v19_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v19.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v19.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v19.csv


# v24 

In [1]:
from models.nba_v24 import Trainer, NBAv24Model

In [7]:
SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

v24_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=100, batch_size=64,
)

model = NBAv24Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v24.pth',weights_only=True
))
model.eval()

all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v24_trainer.mu) / v24_trainer.sigma

    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] 

    # Denormalize
    pred = pred * v24_trainer.sigma.to('cuda') + v24_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v24.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v24.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v24.csv


# v24 

In [9]:
from models.nba_v23 import Trainer, NBAv23Model

In [10]:
SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

v23_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=150, batch_size=64,
)

model = NBAv23Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v23.pth',weights_only=True
))
model.eval()

all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v23_trainer.mu) / v23_trainer.sigma

    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] 

    # Denormalize
    pred = pred * v23_trainer.sigma.to('cuda') + v23_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v23.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v23.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v23.csv


next to try. 

 1. You're not using data augmentation at all.
  Basketball courts are symmetric — you can flip sequences horizontally (x →
  -x), vertically (y → -y), or rotate 180°. That 2-4× more effective training
  data would significantly reduce the Kaggle gap and help generalization. This
  is almost certainly what the top team is doing.

  2. Your local score is inflated — train == val.
  train_path=TRAIN, val_path=TRAIN means the model sees validation sequences
  during training. Your "local 3.83" is partially measuring training data, so
  the real generalization gap is larger than it looks. The ~0.4 local→Kaggle
  gap is a symptom of this.

  3. No ensembling.
  Averaging predictions from v19 + v24 (or more) would smooth out errors. The
  top teams almost certainly ensemble.

  4. Possibly larger capacity.
  state_dim=128 with 2 layers is fairly small. Going to 256 or 3 layers might
  help, especially with augmentation to regularize it.

  Highest-ROI next steps in order:
  1. Add court flip augmentation (x-flip and y-flip during training) — likely
  worth 0.1-0.2 points
  2. Ensemble v19 + v24 predictions — easy, maybe 0.05-0.1
  3. Fix the train/val split so local scores are trustworthy


In [1]:
from models.nba_v31 import Trainer, NBAv31Model

In [8]:
SOLUTIONS_DIR = Path(r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions')
#SOLUTIONS_DIR = r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions'

v31_trainer = Trainer(
    train_path=TRAIN, val_path=TRAIN,
    device='cuda', epochs=150, batch_size=64,
)

model = NBAv31Model(
    state_dim=128, context_size=8, horizon_size=12,
    num_heads=4, num_layers=2,
).to('cuda')
model.load_state_dict(torch.load(
    r'C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\weights\model_v31.pth',weights_only=True
))
model.eval()

all_traj = []
for f in [f for f in os.listdir(TEST) if f.endswith('.pt')]:
    seq = torch.load(os.path.join(TEST, f), weights_only=False)

    seq[:, :, [0, 1]] = (seq[:, :, [0, 1]].clone() - v31_trainer.mu) / v31_trainer.sigma

    X = seq[-8:].unsqueeze(0).to('cuda')  # [1, 8, 11, 4]

    with torch.no_grad():
        pred = model(X)  # [12, 11, 2] 

    # Denormalize
    pred = pred * v31_trainer.sigma.to('cuda') + v31_trainer.mu.to('cuda')
    pred = pred.reshape(12, 11, 2).cpu()  # [12 steps, 11 entities, xy]

    traj = pred.reshape(-1)  # flatten in same order as v2
    all_traj.append([int(f.removesuffix('.pt'))] + traj.tolist())

cols = ['id'] + [f'entity_{i}_time_{t}_{axis}'
                    for t in range(12) for i in range(11) for axis in ['x', 'y']]
pd.DataFrame(all_traj, columns=cols).set_index('id').sort_index().to_csv(
    SOLUTIONS_DIR / 'solution_v31.csv'
)

print("Saved to", SOLUTIONS_DIR / 'solution_v31.csv')

Saved to C:\Users\jan\Downloads\nml_project_proposal\nml_project_proposal\models\solutions\solution_v31.csv
